In [1]:
pip install ioh


[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import numpy as np
import ioh
from ioh import logger
from ACO import ACO
from mmas import mmas, resolve_rho
from mmas_star import run_mmas_star_ioh
import math

#config
DIM = 100
BUDGET = 100000
RUNS = 10
FUNCTIONS = [1, 2, 3, 18, 23, 24, 25]
LOG_DIR = "ioh_log_ACO_MMAS_MMAS_star"


def run_aco(problem, budget, runs=10):
    for r in range(runs):
        log = logger.Analyzer(
            root=LOG_DIR,
            algorithm_name="ACO",
            algorithm_info=f"run_{r}"
        )
        problem.attach_logger(log)

        aco = ACO(n=DIM, num_ants=10, max_iter=budget // 10)
        aco.run(problem)

        problem.detach_logger()


def run_mmas(problem, budget, runs=10):
    for r in range(runs):
        log = logger.Analyzer(
            root=LOG_DIR,
            algorithm_name="MMAS",
            algorithm_info=f"run_{r}"
        )
        problem.attach_logger(log)

        rng = np.random.default_rng(r)
        f_star, curve = mmas(problem, budget, resolve_rho("1/sqrt(n)", DIM), rng)

        problem.detach_logger()



def run_mmas_star(problem, budget, runs=10):
    for r in range(runs):
        log = logger.Analyzer(
            root=LOG_DIR,
            algorithm_name="MMAS*",
            algorithm_info=f"run_{r}"
        )
        problem.attach_logger(log)

        rho = 1.0 / math.sqrt(DIM)
        rng = np.random.default_rng(r)

        problem.reset()
        run_mmas_star_ioh(problem, budget, rho, rng)

        problem.detach_logger()


for fid in FUNCTIONS:
    print(f"Running F{fid}...")

    problem = ioh.get_problem(
        fid, dimension=DIM, instance=1,
        problem_class=ioh.ProblemClass.PBO
    )

    run_aco(problem, BUDGET, RUNS)
    run_mmas(problem, BUDGET, RUNS)
    run_mmas_star(problem, BUDGET, RUNS)

print("Results saved in:", LOG_DIR)

ImportError: cannot import name 'run_mmas_star_ioh' from 'mmas_star' (c:\Users\14035\OneDrive\桌面\code\Assignment 2\Assignment-1-TSP\mmas_star.py)

# Analysis


The ACO algorithm seems to have certain advantages in rapid convergence only in simple functions like F1 and F3, but it almost completely fails in other more complex or deceptive functions. So ACO is only suitable for handling simple problems. The algorithms of MMAS and MMAS* are much better. They can find relatively good results or even optimal solutions in all functions, and at the same time have excellent robustness. Among them, the early performance and convergence speed of MMAS* are even faster than those of MMAS. Overall, MMAS* is the most outstanding algorithm, followed by MMAS, while ACO performs relatively poorly.